<a href="https://colab.research.google.com/github/aryananand0/Math-344-Lab-Project-Fall-2026/blob/main/Notebooks/02_baselines.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Milestone 2: Honest Testing and Baselines

## Part A: Load the MovieLens ratings

In [1]:
import os, urllib.request, zipfile
import numpy as np
import pandas as pd

if not os.path.exists("ml-100k"):
    urllib.request.urlretrieve(
        "https://files.grouplens.org/datasets/movielens/ml-100k.zip",
        "ml-100k.zip")
    zipfile.ZipFile("ml-100k.zip").extractall()

ratings = pd.read_csv(
    "ml-100k/u.data", sep="\t",
    names=["user", "item", "rating", "time"])

print(ratings.head())
print("Number of ratings:", len(ratings))

   user  item  rating       time
0   196   242       3  881250949
1   186   302       3  891717742
2    22   377       1  878887116
3   244    51       2  880606923
4   166   346       1  886397596
Number of ratings: 100000


First I import the libraries I need: `os`, `urllib`, and `zipfile` for downloading and unzipping the data, plus NumPy and Pandas for the math and tables. The `if` statement checks whether the `ml-100k` folder is already there, so the notebook doesn't download the data again every time I run it. Then `pd.read_csv` reads the `u.data` file, which is separated by tabs (`\t`), and I name the four columns user, item (the movie), rating, and time.

Each row is one rating: one user giving one movie a number of stars at a certain time. The output confirms there are 100,000 ratings. This format, with one row per rating, is called "long form." Later I'll turn the training ratings into a user-by-movie matrix.

## Part B: Make an honest train/test split

In [2]:
def split_ratings(ratings, test_frac=0.10, seed=0):
    """Hide ratings without removing a user's or movie's last training rating."""
    rng = np.random.default_rng(seed)
    is_test = np.zeros(len(ratings), dtype=bool)
    user_left = ratings.groupby("user").size().to_dict()
    item_left = ratings.groupby("item").size().to_dict()

    for pos in rng.permutation(len(ratings)):
        row = ratings.iloc[pos]
        user, item = row["user"], row["item"]
        if (rng.random() < test_frac
                and user_left[user] > 1
                and item_left[item] > 1):
            is_test[pos] = True
            user_left[user] -= 1
            item_left[item] -= 1

    return ratings.iloc[~is_test].copy(), ratings.iloc[is_test].copy()

train, test = split_ratings(ratings)
print("train:", len(train), "test:", len(test))
print("test fraction:", round(len(test) / len(ratings), 3))

train: 90032 test: 9968
test fraction: 0.1


This function splits the ratings into a training set and a test set. The test ratings are hidden, so no model ever gets to see them while it's being built.

- `is_test` starts as an array of all `False`, with one entry per rating.
- `user_left` and `item_left` keep track of how many training ratings each user and each movie still has.
- The loop goes through the ratings in a random order using `rng.permutation`.
- Each rating has about a 10% chance (`rng.random() < 0.10`) of being moved to the test set, but only if that user and that movie would still have at least one rating left in training. This makes sure no user or movie ends up with zero training data.
- When a rating gets hidden, I mark it `True` and subtract 1 from both counts.
- At the end, the `~` flips True and False, so `ratings.iloc[~is_test]` gives the training rows and `ratings.iloc[is_test]` gives the test rows.

My split ended up with 90,032 training ratings and 9,968 test ratings, which is about 10%. The seed makes the split the same every time I run it, so every model gets compared on the exact same hidden ratings.

## Part C: Build the training matrix

In [3]:
N_USERS, N_ITEMS = 943, 1682

def to_matrix(df):
    """Return a user-by-movie matrix; missing ratings are np.nan."""
    R = np.full((N_USERS, N_ITEMS), np.nan)
    R[df["user"].to_numpy() - 1,
      df["item"].to_numpy() - 1] = df["rating"].to_numpy()
    return R

R = to_matrix(train)
known = ~np.isnan(R)
print("Matrix shape:", R.shape)
print("Known training ratings:", known.sum())

Matrix shape: (943, 1682)
Known training ratings: 90032


This turns the list of training ratings into a matrix with one row for each user (943) and one column for each movie (1682). I start with a matrix full of `np.nan`, which means "unknown." This is important because a missing rating is not the same as a rating of 0 stars. Then I fill in the ratings that are in the training set.

The user and movie IDs in the data start at 1, but Python counts rows and columns starting at 0, so I subtract 1 from each ID. `known` is a True/False matrix that is True wherever there is a training rating. The output shows the matrix is 943 × 1682 and has 90,032 known ratings, which matches the size of the training set.

In [4]:
truth = test["rating"].to_numpy()
u = test["user"].to_numpy() - 1
i = test["item"].to_numpy() - 1

This sets up the test ratings for scoring. `truth` holds the real star ratings that I hid. `u` and `i` hold the row (user) and column (movie) for each hidden rating, again subtracting 1 to match Python's indexing. This way I can look up each model's prediction at exactly those spots and compare it to `truth`.

## Part D: Score predictions with RMSE

In [5]:
def rmse(pred, truth):
    """Root mean squared error, measured in stars."""
    return np.sqrt(np.mean((pred - truth)**2))

This function measures how far off the predictions are, using root mean squared error (RMSE). For each hidden rating, it:

1. finds the error (prediction minus truth),
2. squares it so positive and negative errors don't cancel out,
3. averages all the squared errors,
4. takes the square root so the answer is back in units of stars.

An RMSE around 1 means the predictions are usually about one star off. A lower RMSE means a more accurate model.

## Part E: Build simple baselines


In [6]:
mu = np.nanmean(R)
user_mean = np.nanmean(R, axis=1)

n_item = known.sum(axis=0)
item_mean = np.full(N_ITEMS, mu)
has_rating = n_item > 0
item_mean[has_rating] = (
    np.nansum(R, axis=0)[has_rating] / n_item[has_rating])

print("Overall average:", round(mu, 3))

Overall average: 3.53


This computes the averages that the simple models use, from the training matrix only. `np.nanmean` takes an average while skipping the `nan` (missing) entries.

- `mu` is the average of all the training ratings, which came out to about 3.53 stars.
- `user_mean` uses `axis=1`, which averages across each row, so I get one average rating for each user.
- `n_item` counts how many training ratings each movie (column) has. I start every movie's average at `mu` as a backup, then replace it with the movie's real average for every movie that has at least one rating. So if a movie had no training ratings, its prediction just falls back to the overall average.

In [7]:
scores = {}
scores["always 3"] = rmse(np.full(len(test), 3.0), truth)
scores["overall average"] = rmse(np.full(len(test), mu), truth)
scores["user average"] = rmse(user_mean[u], truth)
scores["movie average"] = rmse(item_mean[i], truth)

for name, score in scores.items():
    print(f"{name:16s} {score:.4f}")

always 3         1.2516
overall average  1.1349
user average     1.0426
movie average    1.0284


This scores four simple "models" on the hidden test ratings:

- **always 3:** predict 3 stars for every rating (RMSE 1.2516).
- **overall average:** predict `mu` for every rating (RMSE 1.1349).
- **user average:** predict each user's own average rating, using `user_mean[u]` to look it up for each test rating (RMSE 1.0426).
- **movie average:** predict each movie's average rating, using `item_mean[i]` (RMSE 1.0284).

The movie average did the best out of these four. That makes sense, because some movies are just much more liked or disliked than others, and knowing which movie it is tells you a lot about the rating.

## Part F: Combine user and movie effects

In [8]:
def bias_baseline(R):
    """Overall mean plus one user effect and one movie effect."""
    known = ~np.isnan(R)
    mu = np.nanmean(R)
    b_user = np.nanmean(R - mu, axis=1)
    residual = R - mu - b_user[:, None]
    n_item = known.sum(axis=0)
    b_item = np.zeros(R.shape[1])
    has_rating = n_item > 0
    b_item[has_rating] = (
        np.nansum(residual, axis=0)[has_rating] / n_item[has_rating])
    return mu, b_user, b_item

This function builds a model that predicts **rating = μ + b_user + b_item**. The idea is that some users rate everything higher or lower than other people, and some movies are liked by almost everyone, so the model includes both effects.

- `b_user` is how far each user's ratings are from the overall average, on average. A positive value means the user is a generous rater.
- `b_user[:, None]` turns `b_user` into a column so NumPy subtracts each user's effect from every rating in that user's row.
- `residual` is what's left of each rating after taking out the overall average and the user effect.
- `b_item` is the average residual for each movie, which works like that movie's "boost" or "penalty." If a movie has no training ratings, its effect is set to 0.

In [9]:
mu, b_user, b_item = bias_baseline(R)
pred = np.clip(mu + b_user[u] + b_item[i], 1, 5)
scores["user + movie effects"] = rmse(pred, truth)
print(pd.Series(scores, name="test RMSE").round(4).sort_values())

user + movie effects    0.9545
movie average           1.0284
user average            1.0426
overall average         1.1349
always 3                1.2516
Name: test RMSE, dtype: float64


This runs the bias model on the training matrix and predicts every hidden rating by adding the overall average, that user's effect, and that movie's effect. `np.clip(..., 1, 5)` keeps every prediction between 1 and 5 stars, since a rating can't be outside that range.

The combined model got an RMSE of **0.9545**, which is the best of all five models and the only one under 1. Using both the user and the movie together works better than using either one alone. This is just a simple one-pass baseline, but it's the score the SVD model in Milestone 3 has to beat.

## Exercise 1: By Hand, Then by Computer

The matrix I'm working with is

$$
R = \begin{bmatrix} 5 & 3 & ? \\ 4 & ? & 1 \\ ? & 2 & 2 \end{bmatrix}
$$

where the rows are users, the columns are movies, and "?" means the rating is missing. For every average below, I only used the ratings that are actually there and skipped the missing ones.

**Step 1: Overall mean μ**

The known ratings are 5, 3, 4, 1, 2, and 2. There are 6 of them and they add up to 17, so

$$
\mu = \frac{17}{6} \approx 2.8333
$$

**Step 2: User effects**

For each user, I averaged how far their ratings are from μ. This works out to be the same as taking the user's average rating and subtracting μ.

- User 1: average is (5 + 3)/2 = 4, so b_user = 4 − 2.8333 = **1.1667**
- User 2: average is (4 + 1)/2 = 2.5, so b_user = 2.5 − 2.8333 = **−0.3333**
- User 3: average is (2 + 2)/2 = 2, so b_user = 2 − 2.8333 = **−0.8333**

User 1 rates higher than average, and Users 2 and 3 rate lower.

**Step 3: Residuals**

Next I subtracted μ and the user effect from each known rating. Since μ + b_user is just that user's average, I realized I could get each residual by subtracting the user's average from the rating, which made the arithmetic a lot easier. For example, for User 1 and Movie 1: 5 − 2.8333 − 1.1667 = 5 − 4 = 1.

| | Movie 1 | Movie 2 | Movie 3 |
|---|---|---|---|
| User 1 (avg 4) | 5 − 4 = 1 | 3 − 4 = −1 | ? |
| User 2 (avg 2.5) | 4 − 2.5 = 1.5 | ? | 1 − 2.5 = −1.5 |
| User 3 (avg 2) | ? | 2 − 2 = 0 | 2 − 2 = 0 |

**Step 4: Movie effects**

For each movie, I averaged the residuals in its column:

- Movie 1: (1 + 1.5)/2 = **1.25**
- Movie 2: (−1 + 0)/2 = **−0.5**
- Movie 3: (−1.5 + 0)/2 = **−0.75**

**My results**

- μ ≈ 2.8333
- b_user ≈ [1.1667, −0.3333, −0.8333]
- b_item = [1.25, −0.5, −0.75]

**Predicting the missing ratings**

As an extra check, I used prediction = μ + b_user + b_item to fill in the three missing entries:

- User 1, Movie 3: 2.8333 + 1.1667 − 0.75 = **3.25**
- User 2, Movie 2: 2.8333 − 0.3333 − 0.5 = **2.00**
- User 3, Movie 1: 2.8333 − 0.8333 + 1.25 = **3.25**

In [10]:
tiny = np.array([[5, 3, np.nan],
                 [4, np.nan, 1],
                 [np.nan, 2, 2]])

mu_t, b_user_t, b_item_t = bias_baseline(tiny)
print("mu:", round(mu_t, 4))
print("b_user:", b_user_t.round(4))
print("b_item:", b_item_t.round(4))

# check the predictions for the missing spots
print("User 1, Movie 3:", round(mu_t + b_user_t[0] + b_item_t[2], 4))
print("User 2, Movie 2:", round(mu_t + b_user_t[1] + b_item_t[1], 4))
print("User 3, Movie 1:", round(mu_t + b_user_t[2] + b_item_t[0], 4))

mu: 2.8333
b_user: [ 1.1667 -0.3333 -0.8333]
b_item: [ 1.25 -0.5  -0.75]
User 1, Movie 3: 3.25
User 2, Movie 2: 2.0
User 3, Movie 1: 3.25


To check my work, I entered the matrix in NumPy using `np.nan` for the missing ratings and ran it through `bias_baseline`. The output matched my hand calculations exactly: μ ≈ 2.8333, b_user ≈ [1.1667, −0.3333, −0.8333], and b_item = [1.25, −0.5, −0.75]. The three predicted ratings matched too (3.25, 2.00, and 3.25).

Doing this by hand helped me understand the order the function works in. It finds the user effects first, and then it computes the movie effects from whatever is left over after the user effects are taken out.

## Exercise 2: See What Cheating Looks Like

In the code below, I build the matrix from all the ratings, including the test ones, and use it to compute each user's average. This cheating version gets a test RMSE of **1.0322**, while the honest user-average model, which only used the training ratings, got **1.0426**. So yes, the cheating version scores lower.

The problem is that the score is misleading. In the cheating version, every test rating is included in the average that is used to predict it. Each user's average gets pulled a little toward the exact ratings I'm testing on, so the model is basically getting a peek at the answers. This is called data leakage.

The whole point of hiding the test ratings is to see how well a model does on ratings it has never seen, which is what a real recommender has to do when it predicts ratings that don't exist yet. Since the cheating model already saw the test ratings, its RMSE makes it look more accurate than it really is. If I compared it against an honest model, the leaked one could look better just because it cheated. That's why everything, including the averages and later the SVD, has to be computed from the training ratings only.

In [11]:
R_all = to_matrix(ratings)                 # uses ALL ratings, including test ones
cheat_user_mean = np.nanmean(R_all, axis=1)

cheat_score = rmse(cheat_user_mean[u], truth)
print("Honest user-average RMSE: ", round(scores["user average"], 4))
print("Cheating user-average RMSE:", round(cheat_score, 4))

Honest user-average RMSE:  1.0426
Cheating user-average RMSE: 1.0322


## Exercise 3: Is It Luck?

In the code below, I rerun everything with seed 1. When I did this, all five RMSE values changed. This makes sense, because a different random set of ratings gets hidden, so each model is trained and tested on slightly different data. The changes were small, between about 0.003 and 0.022 stars (shown in the `difference` column of the output below). Interestingly, four of the models did a little better with seed 1, but the user + movie effects model did slightly worse, which shows the changes are just random and not one-directional.

The ranking of the models didn't change. For both seeds, the order from best to worst was user + movie effects, movie average, user average, overall average, and always 3.

What this told me is that a single RMSE score has some luck built into it, depending on which ratings end up in the test set. When two models are far apart, the difference is clearly real. For example, the user + movie effects model beat the movie average by about 0.07 stars with seed 0 and about 0.06 with seed 1, which is much bigger than the seed-to-seed changes. But the gap between the movie average and the user average was only about 0.014 to 0.019 stars, which is about the same size as how much the scores moved between seeds (up to 0.022). The movie average happened to win both times, but with a gap that small I wouldn't be fully confident it's better without trying more seeds.

So when I compare recommenders, I need to make sure they're tested on the same split so it's fair. It would also be a good idea to try a few different seeds and make sure one model wins every time. This will matter in Milestone 3. I'll only be able to say the SVD model is actually better if it beats the bias baseline consistently, and by more than the normal variation between seeds

In [12]:
def run_all(seed):
    """Split with a given seed and return the RMSE of all five models."""
    train_s, test_s = split_ratings(ratings, seed=seed)
    R_s = to_matrix(train_s)
    known_s = ~np.isnan(R_s)
    truth_s = test_s["rating"].to_numpy()
    u_s = test_s["user"].to_numpy() - 1
    i_s = test_s["item"].to_numpy() - 1

    mu_s = np.nanmean(R_s)
    user_mean_s = np.nanmean(R_s, axis=1)
    n_item_s = known_s.sum(axis=0)
    item_mean_s = np.full(N_ITEMS, mu_s)
    has_s = n_item_s > 0
    item_mean_s[has_s] = np.nansum(R_s, axis=0)[has_s] / n_item_s[has_s]

    m, bu, bi = bias_baseline(R_s)
    pred_s = np.clip(m + bu[u_s] + bi[i_s], 1, 5)

    return {
        "always 3": rmse(np.full(len(test_s), 3.0), truth_s),
        "overall average": rmse(np.full(len(test_s), mu_s), truth_s),
        "user average": rmse(user_mean_s[u_s], truth_s),
        "movie average": rmse(item_mean_s[i_s], truth_s),
        "user + movie effects": rmse(pred_s, truth_s),
    }

compare = pd.DataFrame({"seed 0": run_all(0), "seed 1": run_all(1)})
compare["difference"] = compare["seed 1"] - compare["seed 0"]
print(compare.round(4).sort_values("seed 0"))

                      seed 0  seed 1  difference
user + movie effects  0.9545  0.9577      0.0032
movie average         1.0284  1.0159     -0.0125
user average          1.0426  1.0345     -0.0081
overall average       1.1349  1.1127     -0.0222
always 3              1.2516  1.2336     -0.0180
